In [9]:
from Bio import SeqIO
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error as rmse
from scipy.stats import pearsonr


models = [
    'PRIME',
    'DeepSTABp',
    'TemBERTure',
    'TemStaPro',
    'ThermoFormer',
    'TmProt',
    'FINE_650M_FULL_MELTOME',
    'FINE_650M_FLIP',
    'FINE_650M_NO_INT',
]

groups = [
    '1_TEST',
    '3_REVERSED',
    '4_SHUFFLED',
    '2_FIRST_100',
]


file = SeqIO.parse('./data/big_tpp.fasta', "fasta")
big_fasta_labels = dict()

for rec in file:
    d = {_n.split('=')[0]: _n.split('=')[-1] for _n in rec.name.split(';')}
    d['tm'] = float(d['tm'])
    d['seq'] = str(rec._seq)  # remove str(..) if you want access to SeqIO.Seq methods
    big_fasta_labels[d['protein_id']] = d

In [10]:
# Example:

big_fasta_labels['A0A061ACF5_fbl-1']

{'protein_id': 'A0A061ACF5_fbl-1',
 'up_id': 'A0A061ACF5',
 'mapped_id': 'A0A061ACF5',
 'mapped_parcID': 'NaN',
 'sp': 'C.elegans',
 'set': 'Meltome',
 'split': 'train',
 'tm': 49.459,
 'seq': 'MRICFLLLAFLVAETFANELTRCCAGGTRHFKNSNTCSSIKSEGTSMTCQRAASICCLRSLLDNACDSGTDIAKEEESCPSNINILGGGLKKECCDCCLLAKDLLNRNEPCVAPVGFSAGCLRSFNKCCNGDIEITHASEIITGRPLNDPHVLHLGDRCASSHCEHLCHDRGGEKVECSCRSGFDLAPDGMACVDIDECATLMDDCLESQRCLNTPGSFKCIRTLSCGTGYAMDSETERCRDVDECNLGSHDCGPLYQCRNTQGSYRCDAKKCGDGELQNPMTGECTSITCPNGYYPKNGMCNDIDECVTGHNCGAGEECVNTPGSFRCQQKGNLCAHGYEVNGATGFCEDVNECTTGIAACEQKCVNIPGSYQCICDRGFALGPDGTKCEDIDECSIWAGSGNDLCMGGCINTKGSYLCQCPPGYKIQPDGRTCVDVDECAMGECAGSDKVCVNTLGSFKCHSIDCPTNYIHDSLNKNQIADGYSCIKVCSTEDTECLGNHTREVLYQFRAVPSLKTIISPIEVSRIVTHMGVPFSVDYNLDYVGQRHFRIVQERNIGIVQLVKPISGPTVETIKVNIHTKSRTGVILAFNEAIIEISVSKYPF'}

In [12]:
# check results

log_NA = False  # Only for TmProt, 25 sequences do not pass conditions proper to the predictor (sequence size, 'X' amino acids, ...)

for mod in models:
    print(mod)
    for group in groups:
        result = pd.read_csv(f'./results/{mod}/{group}.csv') # fetch result file
        if log_NA:
            NAs = result[result['pred'].isna()] # check for NAs (only TmProt)
            print(f'\tWarning: {len(NAs)} NaN in {mod} ({group}): {NAs['name'].tolist()}')
        result = result.dropna()
        predictions = result['pred']
        labels = result['exp_tm']

        # make dict for easy acces
        name_2_label = result.set_index('name').to_dict()['exp_tm']
        name_2_pred = result.set_index('name').to_dict()['pred']
        assert all(name_2_label[n] == big_fasta_labels[n]['tm'] for n in result['name'])

        # print predictions per group
        corr, p = pearsonr(predictions, labels)
        n_rmse = rmse(predictions, labels) / np.std(labels)
        # print(f"\t{group:12s}: pearson = {str(round(corr, 3)):5s}, n_rmse = {str(round(n_rmse, 3)):5s}")
        print(f"{str(round(corr, 2)):5s} / {str(round(n_rmse, 2)):5s}", end = '    &    ')
    print()

PRIME
0.84  / 1.62     &    0.58  / 2.18     &    0.36  / 2.43     &    0.73  / 1.82     &    
DeepSTABp
0.76  / 0.71     &    0.68  / 0.9      &    0.52  / 1.01     &    0.72  / 0.7      &    
TemBERTure
0.84  / 0.56     &    0.72  / 0.77     &    0.64  / 0.92     &    0.76  / 0.66     &    
TemStaPro
0.68  / 1.34     &    0.48  / 1.54     &    0.3   / 1.63     &    0.39  / 1.42     &    
ThermoFormer
0.84  / 0.57     &    0.74  / 0.82     &    0.67  / 0.92     &    0.76  / 0.65     &    
TmProt
0.82  / 0.59     &    0.7   / 0.74     &    0.58  / 0.86     &    0.74  / 0.67     &    
FINE_650M_FULL_MELTOME
0.89  / 0.5      &    0.7   / 0.86     &    0.59  / 0.99     &    0.74  / 0.68     &    
FINE_650M_FLIP
0.85  / 0.56     &    0.71  / 0.92     &    0.6   / 1.07     &    0.74  / 0.68     &    
FINE_650M_NO_INT
0.59  / 0.86     &    0.56  / 0.94     &    0.5   / 1.01     &    0.42  / 0.94     &    
